# Multi-task ST-GNN on METR-LA and PEMS-BAY — Colab runner
Runs the experiments of the paper in stages on one GPU (written for a free T4). **Before anything else:** *Runtime → Change runtime type → GPU*.

| Cells | What | Time on a T4 |
|---|---|---|
| 1–6 | GPU check, Drive, dependencies, code, data, METR-LA adjacency | 5–10 min |
| 7–8 | smoke test on synthetic data; optional pilot (1 seed) | 1 min; about 1 h |
| 9 | **full METR-LA plan**: core (6 seeds) → β grid → extra seeds for the tuned β → Table 11 | about 17 h |
| 9b | optional training-budget check: the five core variants on seed 0 with up to 100 epochs | about 3 h |
| 10–11 | optional PEMS-BAY replication (1 seed) | about 2.7 h |
| 12–13 | day-block bootstrap on seed 0; summary tables | minutes |

Every run is saved as JSON in `MyDrive/mtstgnn/results` and skipped if it already exists. A free Colab session lasts at most about 12 hours: after a disconnect, re-run cells 1–6 and then cell 9 again. Cells run one after another, so do not start another cell while cell 9 is running. A thinned copy of all results (`results_thin.zip`, about 1 MB) is written to `MyDrive/mtstgnn/export` every 15 minutes.

**Status of the results reported in the paper:** the core stage of cell 9 with seeds 0–2 (15 runs) and the day-block bootstrap of cell 12 (seed 0) were run with this notebook on a free Colab T4. Everything else (the β grid, the tuned MT-F, seeds 3–5 of the core variants, the sensitivity and ablation runs, the 100-epoch check and a platform check) was run with `MT-STGNN_Kaggle.ipynb` on a Kaggle notebook with two T4 GPUs, using the same code files plus the queue `kaggle_run.py`. The PEMS-BAY replication (cells 10–11) was not run. Running cell 9 again in a new GPU session continues from the finished runs on Drive (see `COMPUTE.md`).

In [ ]:
# 1. GPU check (must show a GPU; if not, change the runtime type first)
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# 2. Mount Google Drive (a pop-up asks you to allow access: you must click it yourself)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/mtstgnn/results', exist_ok=True)

In [ ]:
# 3. Dependencies (torch is preinstalled in Colab)
!pip -q install tables scipy gdown

In [ ]:
# 4. Unpack the code package (embedded in this notebook)
import base64, io, zipfile
B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))).extractall('/content')
%cd /content/mtstgnn
!ls

In [ ]:
# 5. METR-LA and PEMS-BAY data. The DCRNN authors publish metr-la.h5 and pems-bay.h5 in a Google Drive folder
#    (link in the DCRNN README: https://drive.google.com/open?id=10FOTa6HXPqX8Pf5WRoRwcFnW9BrNZEIX).
#    A copy is kept in MyDrive/mtstgnn/data, so a restarted session does not depend on that folder's download quota.
import os, shutil, subprocess, glob
CACHE = '/content/drive/MyDrive/mtstgnn/data'
os.makedirs('data/METR-LA', exist_ok=True); os.makedirs('/content/dcrnn_data', exist_ok=True); os.makedirs(CACHE, exist_ok=True)
find = lambda f: (glob.glob('/content/dcrnn_data/**/' + f, recursive=True) or [None])[0]
for f in ['metr-la.h5', 'pems-bay.h5']:
    if find(f) is None and os.path.exists(CACHE + '/' + f):
        shutil.copy(CACHE + '/' + f, '/content/dcrnn_data/' + f); print('copied from the Drive cache:', f)
if find('metr-la.h5') is None or find('pems-bay.h5') is None:
    r = subprocess.run("gdown --folder https://drive.google.com/drive/folders/10FOTa6HXPqX8Pf5WRoRwcFnW9BrNZEIX -O /content/dcrnn_data",
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
for f in ['metr-la.h5', 'pems-bay.h5']:
    if find(f) is not None and not os.path.exists(CACHE + '/' + f):
        shutil.copy(find(f), CACHE + '/' + f); print('saved to the Drive cache:', f)
if find('metr-la.h5') is not None:
    shutil.copy(find('metr-la.h5'), 'data/METR-LA/metr-la.h5'); print('copied', find('metr-la.h5'))
else:
    print("\nDownload failed (Drive quota is common). Open the link above, add the folder to your Drive, then copy metr-la.h5 "
          "and pems-bay.h5 from MyDrive into /content/mtstgnn/data/METR-LA/ and /content/dcrnn_data/ (e.g. !cp /content/drive/MyDrive/<where>/metr-la.h5 data/METR-LA/).")

In [ ]:
# 6. METR-LA adjacency (adj_mx.pkl). Built here from two small DATA files of the DCRNN repo; no code from that repo is run.
# Same rule as DCRNN's gen_adj_mx: Gaussian kernel exp(-(d/sigma)^2), sigma = std of the road distances, entries below 0.1 set to 0.
import os, pickle, urllib.request
import numpy as np, pandas as pd
base = 'https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/'
os.makedirs('/content/sg', exist_ok=True)
for f in ['graph_sensor_ids.txt', 'distances_la_2012.csv']:
    urllib.request.urlretrieve(base + f, '/content/sg/' + f)
ids = open('/content/sg/graph_sensor_ids.txt').read().strip().split(',')
ind = {s: i for i, s in enumerate(ids)}
dist = np.full((len(ids), len(ids)), np.inf, dtype=np.float32)
for a, b, c in pd.read_csv('/content/sg/distances_la_2012.csv', dtype={'from': 'str', 'to': 'str'}).values[:, :3]:
    if a in ind and b in ind:
        dist[ind[a], ind[b]] = float(c)
std = dist[~np.isinf(dist)].std()
adj = np.exp(-np.square(dist / std)); adj[adj < 0.1] = 0
os.makedirs('data/METR-LA', exist_ok=True)
pickle.dump([ids, ind, adj], open('data/METR-LA/adj_mx.pkl', 'wb'))
print(len(ids), 'sensors | non-zero entries', int((adj > 0).sum()), '| sigma', float(std))
print(os.listdir('data/METR-LA'))

In [ ]:
# 7. SMOKE TEST on synthetic data (about a minute). If this fails, copy the whole error and send it back.
!bash run_plan.sh smoke

In [ ]:
# 8. PILOT (optional): one seed of the core stage (R, C, MT-F, MT-U, MT-P), float16, about 1 hour. Check the numbers before launching the full plan.
import os
os.environ['OUT'] = '/content/drive/MyDrive/mtstgnn/results'
os.environ['SEEDS'] = '0'
os.environ['AMP'] = '1'   # float16 encoder: about 1.75x faster
!bash run_plan.sh core metr-la data/METR-LA

In [ ]:
# 9. FULL PLAN, unattended (about 17 h on a T4; every finished run is skipped, so this cell can simply be run again).
# core: 6 seeds of R, C, MT-F (beta 1), MT-U, MT-P | grid: beta in {0.25, 0.5, 2, 4} on seeds 0-2, then the tuned beta on seeds 3-5 | sens: Table 11 on seed 0.
# A thinned copy of all results is written to MyDrive/mtstgnn/export/results_thin.zip every 15 minutes (and after every stage).
import os, re, glob, json, sys, time, shutil, zipfile, threading, platform, hashlib, subprocess
import numpy as np, pandas as pd
os.chdir('/content/mtstgnn')   # the code directory of cell 4 (a restarted runtime starts in /content)
OUT = '/content/drive/MyDrive/mtstgnn/results'
EXP = '/content/drive/MyDrive/mtstgnn/export'
os.makedirs(OUT, exist_ok=True); os.makedirs(EXP, exist_ok=True)
os.environ['OUT'] = OUT
os.environ['AMP'] = '1'
subprocess.run("pkill -f '[r]un_plan.sh'; sleep 1; pkill -f '[t]rain.py'; sleep 3; pkill -9 -f '[t]rain.py'; true", shell=True)   # leftovers of an interrupted run

def export_results():
    """One zip with every result JSON (gradient-cosine log thinned to every 20th step) plus the dataset/environment files."""
    n, bad, tmp = 0, [], '/content/results_thin.zip'
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(glob.glob(OUT + '/*.json')):
            try:
                r = json.load(open(p))
                if 'grad_cos' in r:
                    r['grad_cos'] = [e for e in r['grad_cos'] if e[1] % 20 == 0]
                z.writestr(os.path.basename(p), json.dumps(r)); n += 1
            except Exception:
                bad.append(os.path.basename(p))      # a file that is being written right now
        for p in sorted(glob.glob(EXP + '/*.json')):
            z.write(p, os.path.basename(p))
    shutil.copyfile(tmp, EXP + '/results_thin.zip')
    return n, bad

def dataset_stats(path):
    from data import IGNORE, free_flow_speed, make_labels, split_ranges
    def shares(lab):
        lab = lab[lab != IGNORE]
        return [float('nan')] * 3 if lab.size == 0 else (100.0 * np.bincount(lab, minlength=3)[:3] / lab.size).tolist()
    df = pd.read_hdf(path)
    speed = df.values.astype(np.float32); idx = df.index
    tod = ((idx.hour * 60 + idx.minute).values / 1440.0).astype(np.float32)
    T, N = speed.shape
    _, tr_end = split_ranges(T, 12, 12); va_end = int(T * 0.8)
    vf = free_flow_speed(speed, tod, tr_end); lab = make_labels(speed, vf); obs = speed > 0
    return dict(n_sensors=int(N), n_steps=int(T), start=str(idx[0]), end=str(idx[-1]), days=float(T / 288.0),
                missing_pct=float(100.0 * (~obs).mean()), ignored_label_pct=float(100.0 * (lab == IGNORE).mean()),
                level_pct_all=shares(lab), level_pct_train=shares(lab[:tr_end]), level_pct_val=shares(lab[tr_end:va_end]),
                level_pct_test=shares(lab[va_end:]), speed_mean_train=float(speed[:tr_end][obs[:tr_end]].mean()),
                speed_std_train=float(speed[:tr_end][obs[:tr_end]].std()), v_free_median=float(np.median(vf)),
                v_free_min=float(vf.min()), v_free_max=float(vf.max()),
                split_steps=dict(train=int(tr_end), val=int(va_end - tr_end), test=int(T - va_end)),
                test_days=float((T - va_end) / 288.0))

def setup_files():
    import torch, scipy
    env = dict(time=time.strftime('%Y-%m-%d %H:%M:%S'), python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
               scipy=scipy.__version__, torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, platform=platform.platform(),
               code_sha12={f: hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12]
                           for f in ['train.py', 'model.py', 'data.py', 'metrics.py', 'posthoc.py', 'run_plan.sh']})
    env['nvidia_smi'] = subprocess.run('nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader',
                                       shell=True, capture_output=True, text=True).stdout.strip()
    json.dump(env, open(EXP + '/env.json', 'w'), indent=1)
    print('env:', json.dumps({k: v for k, v in env.items() if k != 'code_sha12'}), flush=True)
    for tag, pat in [('metr', 'data/METR-LA/metr-la.h5'), ('pems', '/content/dcrnn_data/**/pems-bay.h5')]:
        try:
            f = (glob.glob(pat, recursive=True) or [None])[0]
            if f is None:
                print('dataset stats', tag, ': file not found', flush=True); continue
            st = dataset_stats(f); st['file'] = f
            json.dump(st, open(EXP + f'/ds_{tag}.json', 'w'), indent=1)
            print('dataset stats', tag, json.dumps(st), flush=True)
        except Exception as e:
            print('dataset stats', tag, 'FAILED', repr(e), flush=True)

def safe_export(tag=''):
    try:
        n, bad = export_results()
        print('[export %s %s] %d files%s' % (tag, time.strftime('%H:%M'), n, (' | skipped ' + str(bad)) if bad else ''), flush=True)
    except Exception as e:
        print('[export failed]', repr(e), flush=True)

def exporter(period=900):
    while True:
        time.sleep(period)
        safe_export('periodic')

def stage(name, seeds, grid=None):
    os.environ['SEEDS'] = seeds
    if grid:
        os.environ['GRID'] = grid
    print('=== stage', name, '| seeds', seeds, '| grid', grid, '|', time.strftime('%H:%M'), flush=True)
    get_ipython().system(f'bash run_plan.sh {name} metr-la data/METR-LA')
    safe_export('after stage')

def tuned_beta():
    runs = {}
    for p in glob.glob(OUT + '/mtf_b*_s*.json'):
        m = re.match(r'.*/mtf_b([0-9.]+)_s(\d+)\.json$', p)
        if m:
            runs.setdefault(m.group(1), {})[int(m.group(2))] = min(h['val'] for h in json.load(open(p))['history'])
    common = sorted(set.intersection(*[set(v) for v in runs.values()]))
    score = {b: float(np.mean([v[s] for s in common])) for b, v in runs.items()}
    print('validation loss by beta, seeds', common, score, flush=True)
    return min(score, key=score.get)

# ---- start
try:
    setup_files()
except Exception as e:
    print('setup_files FAILED', repr(e), flush=True)
safe_export('start')
if not any(t.name == 'result-exporter' and t.is_alive() for t in threading.enumerate()):
    threading.Thread(target=exporter, name='result-exporter', daemon=True).start()

# ---- stages
stage('core', '0 1 2 3 4 5')
stage('grid', '0 1 2', '0.25 0.5 2 4')
best = tuned_beta()
print('tuned MT-F beta:', best, flush=True)
if best != '1':
    stage('grid', '3 4 5', best)
stage('sens', '0')
print('ALL STAGES DONE', time.strftime('%H:%M'), flush=True)

In [ ]:
# 9b. Training-budget check (optional, run after cell 9): the five core variants on seed 0 with at most 100 epochs instead of 50 (about 3 h on a T4)
import os, glob, json, time, zipfile, shutil
os.chdir('/content/mtstgnn')
OUT_E = '/content/drive/MyDrive/mtstgnn/results_e100'
EXP = '/content/drive/MyDrive/mtstgnn/export'
os.makedirs(OUT_E, exist_ok=True); os.makedirs(EXP, exist_ok=True)
os.environ.update(OUT=OUT_E, AMP='1', SEEDS='0', EPOCHS='100')
print('=== budget check, 100 epochs |', time.strftime('%H:%M'), flush=True)
get_ipython().system('bash run_plan.sh core metr-la data/METR-LA')
with zipfile.ZipFile('/content/results_e100_thin.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(OUT_E + '/*.json')):
        r = json.load(open(p))
        if 'grad_cos' in r:
            r['grad_cos'] = [e for e in r['grad_cos'] if e[1] % 20 == 0]
        z.writestr(os.path.basename(p), json.dumps(r))
shutil.copyfile('/content/results_e100_thin.zip', EXP + '/results_e100_thin.zip')
print('BUDGET CHECK DONE', time.strftime('%H:%M'), flush=True)

In [ ]:
# 10. PEMS-BAY adjacency (adj_mx_bay.pkl): a DATA file of the DCRNN repo (about 1 MB). It is read with a restricted unpickler
# (only numpy array rebuilding is allowed), checked against the sensor order of pems-bay.h5 and re-saved as a clean pickle.
import os, pickle, shutil, glob, urllib.request
import numpy as np, pandas as pd
os.chdir('/content/mtstgnn')
class SafeUnpickler(pickle.Unpickler):
    OK = {('numpy.core.multiarray', '_reconstruct'), ('numpy._core.multiarray', '_reconstruct'), ('numpy', 'ndarray'),
          ('numpy', 'dtype'), ('_codecs', 'encode'), ('numpy.core.multiarray', 'scalar'), ('numpy._core.multiarray', 'scalar')}
    def find_class(self, module, name):
        if (module, name) in self.OK:
            return super().find_class(module, name)
        raise pickle.UnpicklingError('blocked: %s.%s' % (module, name))
os.makedirs('data/PEMS-BAY', exist_ok=True)
shutil.copy(glob.glob('/content/dcrnn_data/**/pems-bay.h5', recursive=True)[0], 'data/PEMS-BAY/pems-bay.h5')
raw = 'data/PEMS-BAY/adj_mx_bay_raw.pkl'
urllib.request.urlretrieve('https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/adj_mx_bay.pkl', raw)
with open(raw, 'rb') as f:
    ids, ind, adj = SafeUnpickler(f, encoding='latin1').load()
adj = np.asarray(adj, dtype=np.float32)
cols = [str(c) for c in pd.read_hdf('data/PEMS-BAY/pems-bay.h5').columns]
assert [str(i) for i in ids] == cols, 'sensor order of the adjacency differs from the h5 file'
pickle.dump([ids, ind, adj], open('data/PEMS-BAY/adj_mx_bay.pkl', 'wb'))
print(len(ids), 'sensors | shape', adj.shape, '| non-zero entries', int((adj > 0).sum()), '| max', float(adj.max()))

In [ ]:
# 11. PEMS-BAY replication: R, C, MT-F (beta chosen on METR-LA validation data), MT-P. About 2.7 h per seed on a T4 (float16).
import os, re, glob, json, time, zipfile, shutil
import numpy as np
OUT_M, OUT_P, EXP = '/content/drive/MyDrive/mtstgnn/results', '/content/drive/MyDrive/mtstgnn/results_pems', '/content/drive/MyDrive/mtstgnn/export'
os.chdir('/content/mtstgnn')
os.makedirs(OUT_P, exist_ok=True); os.makedirs(EXP, exist_ok=True)
def beta_from_metr():
    runs = {}
    for p in glob.glob(OUT_M + '/mtf_b*_s*.json'):
        m = re.match(r'.*/mtf_b([0-9.]+)_s(\d+)\.json$', p)
        if m:
            runs.setdefault(m.group(1), {})[int(m.group(2))] = min(h['val'] for h in json.load(open(p))['history'])
    common = sorted(set.intersection(*[set(v) for v in runs.values()]))
    score = {b: float(np.mean([v[s] for s in common])) for b, v in runs.items()}
    return min(score, key=score.get)
def zip_pems():
    with zipfile.ZipFile('/content/results_pems_thin.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(glob.glob(OUT_P + '/*.json')):
            try:
                r = json.load(open(p))
                if 'grad_cos' in r:
                    r['grad_cos'] = [e for e in r['grad_cos'] if e[1] % 20 == 0]
                z.writestr(os.path.basename(p), json.dumps(r))
            except Exception:
                pass
    shutil.copyfile('/content/results_pems_thin.zip', EXP + '/results_pems_thin.zip')
best = beta_from_metr()
print('PEMS-BAY: MT-F uses beta =', best, '(chosen on METR-LA validation loss)', time.strftime('%H:%M'), flush=True)
os.environ.update(OUT=OUT_P, AMP='1', SEEDS='0', BETA=best)
get_ipython().system('bash run_plan.sh pems pems-bay data/PEMS-BAY')
zip_pems()
print('PEMS-BAY DONE', time.strftime('%H:%M'), flush=True)

In [ ]:
# 12. Day-block bootstrap of the paired differences on the seed-0 test predictions (run after the beta grid has finished)
import os, re, glob, json
import numpy as np
os.chdir('/content/mtstgnn')
OUT, EXP = '/content/drive/MyDrive/mtstgnn/results', '/content/drive/MyDrive/mtstgnn/export'
sc = {}
for p in glob.glob(OUT + '/mtf_b*_s*.json'):
    m = re.match(r'.*/mtf_b([0-9.]+)_s(\d+)\.json$', p)
    if m:
        sc.setdefault(m.group(1), {})[int(m.group(2))] = min(h['val'] for h in json.load(open(p))['history'])
common = sorted(set.intersection(*[set(v) for v in sc.values()]))
best = min(sc, key=lambda b: np.mean([sc[b][s] for s in common]))
print('tuned beta', best, flush=True)
get_ipython().system(f'python bootstrap.py --dir {OUT} --best mtf_b{best} --out {EXP}/bootstrap_metr.json')

In [ ]:
# 13. Summary tables (mean ± sd over seeds, H1-H4 checks) -> MyDrive/mtstgnn/results/summary.md
import os
os.chdir('/content/mtstgnn')
os.environ['OUT'] = '/content/drive/MyDrive/mtstgnn/results'
!bash run_plan.sh summary
print(open('/content/drive/MyDrive/mtstgnn/results/summary.md').read())

**Keep / send back:** `MyDrive/mtstgnn/export/results_thin.zip` (every number of the paper's tables) and `MyDrive/mtstgnn/results/summary.md`.
Do not edit numbers by hand: `python aggregate.py <results dir> --json out.json` regenerates the tables and the H1–H4 checks from the run files.